# Вычисление test-метрик по сохранённым чекпойнтам

Три сценария:
- **E1** — `AHTransformer` (stage 1) на BAH test
- **E2** — `EmotionTransformer` (stage 1) на MOSEI test
- **E3–E6** — `FusionTransformer` (stage 2) на (MOSEI + BAH) test, по сидам

> Запускать из `mehr/` или из `mehr/notebooks/` — пути строятся автоматически.

In [1]:
import sys, json
from pathlib import Path
from types import SimpleNamespace

# Определяем PKG_DIR независимо от cwd
# Ноутбук лежит в mehr/notebooks/ → PKG_DIR = mehr/
_nb_dir = Path(".").resolve()
PKG_DIR = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset
import numpy as np

from models import EmotionTransformer, AHTransformer, FusionTransformer
from data.datasets import DatasetEmotionAHFusion, custom_collate_fn
from training.metrics import predict_emotions, mf1, uar, mwacc, wf1
from training.epochs import eval_one_epoch

RESULTS = PKG_DIR / "results"
CACHE   = PKG_DIR / "data" / "embeddings_cache"
MOSEI   = PKG_DIR / "data" / "raw" / "EAAI" / "CMU-MOSEI"
BAH     = PKG_DIR / "data" / "raw" / "bah_data" / "split"
BATCH   = 32
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"PKG_DIR: {PKG_DIR}")
print(f"Device:  {device}")

/Users/ildarkhamaganov/Documents/HSE/NIR/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PKG_DIR: /Users/ildarkhamaganov/Documents/HSE/NIR/mehr
Device:  mps


## Вспомогательные функции

In [7]:
def load_cfg(path):
    with open(path) as f:
        return SimpleNamespace(**json.load(f))


def make_test_loader_fusion(cfg):
    """Test-loader для стадии 2 (MOSEI + BAH), только закешированные эмбеддинги."""
    emo_ds = DatasetEmotionAHFusion(
        dataset="CMU-MOSEI", part="test",
        path=cfg.mosei_path,
        path_to_emb=getattr(cfg, "mosei_test_emb", None),
        model=cfg.encoder_model,
    )
    ah_ds = DatasetEmotionAHFusion(
        dataset="BAH", part="test",
        path=cfg.bah_path,
        path_to_emb=getattr(cfg, "bah_test_emb", None),
        model=cfg.encoder_model,
    )
    return DataLoader(ConcatDataset([emo_ds, ah_ds]), batch_size=cfg.batch_size,
                      shuffle=False, collate_fn=custom_collate_fn)


def build_fusion_model(cfg, seed):
    """Строит FusionTransformer и загружает чекпойнт.

    Имена ключей в чекпойнтах (per_proj / ah_model.per_proj) уже совпадают
    с текущей AHTransformer — ремап ключей НЕ нужен.
    """
    emo_model = EmotionTransformer(
        input_dim_emotion=384, hidden_dim=256, out_features=256,
        num_transformer_heads=4, tr_layer_number=3, dropout=0.0,
    ).to(device)
    ah_model = AHTransformer(
        input_dim_ah=384, hidden_dim=512, out_features=128,
        num_transformer_heads=8, tr_layer_number=1, dropout=0.2,
    ).to(device)

    emo_model.load_state_dict(torch.load(cfg.emo_model_path, map_location=device))
    ah_model.load_state_dict(torch.load(cfg.ah_model_path, map_location=device))
    emo_model.eval(); ah_model.eval()

    model = FusionTransformer(
        emo_model=emo_model, ah_model=ah_model,
        hidden_dim=cfg.hidden_dim, out_features=cfg.out_features,
        num_transformer_heads=cfg.num_transformer_heads,
        tr_layer_number=cfg.tr_layer_number, dropout=cfg.dropout,
        unfreeze_encoders=False,
    ).to(device)

    exp = Path(cfg.output_path).stem.rsplit(f"_seed{seed}", 1)[0]
    fusion_sd = torch.load(RESULTS / f"{exp}_seed{seed}.pt", map_location=device)
    model.load_state_dict(fusion_sd)
    return model

## E1 — AHTransformer (stage 1) на BAH test

In [3]:
ah_ds = DatasetEmotionAHFusion(
    dataset="BAH", part="test",
    path=str(BAH),
    path_to_emb=str(CACHE / "BAH_test_bge-small_embeddings.pkl"),
    model="bge-small",
)
ah_loader = DataLoader(ah_ds, batch_size=BATCH, shuffle=False,
                       collate_fn=custom_collate_fn)

ah_model = AHTransformer(
    input_dim_ah=384, hidden_dim=512, out_features=128,
    num_transformer_heads=8, tr_layer_number=1, dropout=0.2,
).to(device)
ah_model.load_state_dict(torch.load(RESULTS / "Transformer_bge-small_ah.pt",
                                    map_location=device))
ah_model.eval()

preds_e1, trues_e1 = [], []
with torch.no_grad():
    for batch in ah_loader:
        if batch is None: continue
        emb    = batch["text_embedding"].to(device)
        labels = batch["ah_labels"].to(device)
        valid  = ~torch.isnan(labels)
        if not valid.any(): continue
        logits = ah_model(ah_input=emb)["ah_scores"]
        _, pred = torch.max(logits[valid], dim=1)
        preds_e1.extend(pred.cpu().numpy())
        trues_e1.extend(labels[valid].long().cpu().numpy())

t2d = [[t] for t in trues_e1]; p2d = [[p] for p in preds_e1]
e1_results = {
    "ah_mf1": mf1(t2d, p2d),
    "ah_uar": uar(t2d, p2d),
    "ah_wf1": wf1(t2d, p2d),
    "n":      len(trues_e1),
}
print(f"E1  AHTransformer → BAH test  (n={e1_results['n']})")
print(f"  ah_mf1  = {e1_results['ah_mf1']:.4f}")
print(f"  ah_uar  = {e1_results['ah_uar']:.4f}")
print(f"  ah_wf1  = {e1_results['ah_wf1']:.4f}")

Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/BAH_test_bge-small_embeddings.pkl...


/var/folders/pn/8q4knbjn5rj60_0r74hvf_dw0000gn/T/ipykernel_62980/104523755.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ah_model.load_state_dict(torch.load(RESULTS /

E1  AHTransformer → BAH test  (n=525)
  ah_mf1  = 0.6853
  ah_uar  = 0.6868
  ah_wf1  = 0.6982


## E2 — EmotionTransformer (stage 1) на MOSEI test

In [4]:
emo_ds = DatasetEmotionAHFusion(
    dataset="CMU-MOSEI", part="test",
    path=str(MOSEI),
    path_to_emb=str(CACHE / "CMU-MOSEI_test_eaai_bge-small_embeddings.pkl"),
    model="bge-small",
)
emo_loader = DataLoader(emo_ds, batch_size=BATCH, shuffle=False,
                        collate_fn=custom_collate_fn)

emo_model_orig = EmotionTransformer(
    input_dim_emotion=384, hidden_dim=256, out_features=256,
    num_transformer_heads=4, tr_layer_number=3, dropout=0.0,
).to(device)
emo_model_mine = EmotionTransformer(
    input_dim_emotion=384, hidden_dim=256, out_features=256,
    num_transformer_heads=4, tr_layer_number=3, dropout=0.0,
).to(device)
emo_model_mine.load_state_dict(torch.load(RESULTS / "Transformer_bge-small_emotion.pt",
                                     map_location=device))
emo_model_orig.load_state_dict(torch.load("/Users/ildarkhamaganov/Documents/HSE/NIR/stage1_ssl_mepr/Transformer_bge-small_emotion.pt",
                                     map_location=device))
for emo_model in [emo_model_orig, emo_model_mine]:
    emo_model.eval()

    preds_e2, trues_e2 = [], []
    with torch.no_grad():
        for batch in emo_loader:
            if batch is None: continue
            emb    = batch["text_embedding"].to(device)
            labels = batch["emo_labels"].to(device)
            valid  = ~torch.isnan(labels[:, 0])
            if not valid.any(): continue
            logits = emo_model(emotion_input=emb)["emotion_logits"]
            preds_e2.extend(predict_emotions(logits[valid]))
            trues_e2.extend((labels[valid][:, 1:] > 0).long().cpu().numpy())

    e2_results = {
        "emo_mf1":   mf1(trues_e2, preds_e2),
        "emo_uar":   uar(trues_e2, preds_e2),
        "emo_mwacc": mwacc(trues_e2, preds_e2),
        "n":         len(trues_e2),
    }
    print(f"E2  EmotionTransformer → MOSEI test  (n={e2_results['n']})")
    print(f"  emo_mf1   = {e2_results['emo_mf1']:.4f}")
    print(f"  emo_uar   = {e2_results['emo_uar']:.4f}")
    print(f"  emo_mwacc = {e2_results['emo_mwacc']:.4f}")

Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/CMU-MOSEI_test_eaai_bge-small_embeddings.pkl...


/var/folders/pn/8q4knbjn5rj60_0r74hvf_dw0000gn/T/ipykernel_62980/221415063.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emo_model_mine.load_state_dict(torch.load(RES

E2  EmotionTransformer → MOSEI test  (n=4653)
  emo_mf1   = 0.5848
  emo_uar   = 0.6441
  emo_mwacc = 0.6441
E2  EmotionTransformer → MOSEI test  (n=4653)
  emo_mf1   = 0.5848
  emo_uar   = 0.6441
  emo_mwacc = 0.6441


## E3–E6 — FusionTransformer (stage 2) по сидам

Укажите название эксперимента и список сидов для пересчёта.

In [5]:
EXP_NAME = "E3_fusion_no_ssl"     # или E4_fusion_ssl, E5_fusion_gradnorm, E6_fusion_ssl_unfreeze
SEEDS    = [42, 0, 123, 1, 2, 7, 13, 21, 100, 2024]
STAGE2_METRICS = ["emo_mf1", "emo_uar", "emo_mwacc", "ah_mf1", "ah_uar", "ah_wf1", "overall_f1"]

In [8]:
criterion = nn.CrossEntropyLoss()   # без весов — метрики не зависят от лосса
results_by_seed = {}

for seed in SEEDS:
    cfg_path = RESULTS / f"{EXP_NAME}_seed{seed}.config.json"
    pt_path  = RESULTS / f"{EXP_NAME}_seed{seed}.pt"
    if not cfg_path.exists() or not pt_path.exists():
        print(f"[seed={seed}] пропуск: файлы не найдены")
        continue

    cfg  = load_cfg(cfg_path)
    loader = make_test_loader_fusion(cfg)
    model  = build_fusion_model(cfg, seed)
    model.eval()

    m = eval_one_epoch(model, loader, criterion, criterion, device)
    results_by_seed[seed] = {k: m[k] for k in STAGE2_METRICS}
    print(
        f"seed={seed:4d}  "
        f"emo_mf1={m['emo_mf1']:.4f}  emo_mwacc={m['emo_mwacc']:.4f}  "
        f"ah_mf1={m['ah_mf1']:.4f}  ah_wf1={m['ah_wf1']:.4f}  "
        f"overall={m['overall_f1']:.4f}"
    )

print()
for metric in STAGE2_METRICS:
    vals = [results_by_seed[s][metric] for s in results_by_seed]
    if vals:
        print(f"{metric:<14}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/CMU-MOSEI_test_eaai_bge-small_embeddings.pkl...
Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/BAH_test_bge-small_embeddings.pkl...


/var/folders/pn/8q4knbjn5rj60_0r74hvf_dw0000gn/T/ipykernel_62980/282241009.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emo_model.load_state_dict(torch.load(cfg.emo_

seed=  42  emo_mf1=0.5836  emo_mwacc=0.5837  ah_mf1=0.6800  ah_wf1=0.6874  overall=0.6318
Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/CMU-MOSEI_test_eaai_bge-small_embeddings.pkl...
Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/BAH_test_bge-small_embeddings.pkl...


/var/folders/pn/8q4knbjn5rj60_0r74hvf_dw0000gn/T/ipykernel_62980/282241009.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emo_model.load_state_dict(torch.load(cfg.emo_

seed=   0  emo_mf1=0.5852  emo_mwacc=0.5856  ah_mf1=0.6736  ah_wf1=0.6798  overall=0.6294
Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/CMU-MOSEI_test_eaai_bge-small_embeddings.pkl...
Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/BAH_test_bge-small_embeddings.pkl...


...

seed=  21  emo_mf1=0.5867  emo_mwacc=0.5878  ah_mf1=0.6836  ah_wf1=0.6927  overall=0.6351
Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/CMU-MOSEI_test_eaai_bge-small_embeddings.pkl...
Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/BAH_test_bge-small_embeddings.pkl...


/var/folders/pn/8q4knbjn5rj60_0r74hvf_dw0000gn/T/ipykernel_62980/282241009.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emo_model.load_state_dict(torch.load(cfg.emo_

seed= 100  emo_mf1=0.5825  emo_mwacc=0.5860  ah_mf1=0.6729  ah_wf1=0.6798  overall=0.6277
Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/CMU-MOSEI_test_eaai_bge-small_embeddings.pkl...
Loading embeddings from /Users/ildarkhamaganov/Documents/HSE/NIR/mehr/data/embeddings_cache/BAH_test_bge-small_embeddings.pkl...


/var/folders/pn/8q4knbjn5rj60_0r74hvf_dw0000gn/T/ipykernel_62980/282241009.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emo_model.load_state_dict(torch.load(cfg.emo_

seed=2024  emo_mf1=0.5811  emo_mwacc=0.5838  ah_mf1=0.6789  ah_wf1=0.6873  overall=0.6300

emo_mf1       : 0.5840 ± 0.0017
emo_uar       : 0.5853 ± 0.0019
emo_mwacc     : 0.5853 ± 0.0019
ah_mf1        : 0.6756 ± 0.0089
ah_uar        : 0.6873 ± 0.0066
ah_wf1        : 0.6831 ± 0.0105
overall_f1    : 0.6298 ± 0.0049


## Сводная таблица (stage-1 vs stage-2 baseline)

In [9]:
import pandas as pd

rows = [
    {"Модель": "E1 AHTransformer (stage 1)",
     "ah_mf1": e1_results["ah_mf1"], "ah_uar": e1_results["ah_uar"],
     "ah_wf1": e1_results["ah_wf1"]},
    {"Модель": "E2 EmotionTransformer (stage 1)",
     "emo_mf1": e2_results["emo_mf1"], "emo_uar": e2_results["emo_uar"],
     "emo_mwacc": e2_results["emo_mwacc"]},
]

if results_by_seed:
    vals = list(results_by_seed.values())
    rows.append({
        "Модель": f"{EXP_NAME} mean±std",
        "emo_mf1":   np.mean([v["emo_mf1"]   for v in vals]),
        "emo_mwacc": np.mean([v["emo_mwacc"] for v in vals]),
        "ah_mf1":    np.mean([v["ah_mf1"]    for v in vals]),
        "ah_wf1":    np.mean([v["ah_wf1"]    for v in vals]),
        "overall_f1": np.mean([v["overall_f1"] for v in vals]),
    })

pd.DataFrame(rows).set_index("Модель").round(4)

,ah_mf1,ah_uar,ah_wf1,emo_mf1,emo_uar,emo_mwacc,overall_f1
Модель,,,,,,,
E1 AHTransformer (stage 1),0.6853,0.6868,0.6982,NaN,NaN,NaN,NaN
E2 EmotionTransformer (stage 1),NaN,NaN,NaN,0.5848,0.6441,0.6441,NaN
E3_fusion_no_ssl mean±std,0.6756,NaN,0.6831,0.5840,NaN,0.5853,0.6298
